# Entrega — Agente 10-K (`responder` / `evaluar`)

Notebook minimo de entrega. Pensado para ejecutarse de arriba a abajo desde un clon limpio del repositorio, con este archivo en la **raiz del repo** (junto a `common/`).

**Requisitos previos** (ver tambien la seccion "Ejecucion desde un clon limpio" del README):

1. Dependencias instaladas: `pip install -r common/requirements.txt`
2. `OPENROUTER_API_KEY` definida: copia `.env.example` (en la raiz del repo) a `.env` y rellena tu clave real. `.env` esta en `.gitignore`, nunca se sube a Git.
3. El dataset oficial de la practica colocado segun se documenta en `common/README.md` (por defecto, como `dataset/` junto a este repo, o mediante la variable `MIAX_DATASET_DIR`).

Este notebook **no** reconstruye indices FAISS ni ejecuta baterias de preguntas de forma automatica: las celdas de ejemplo hacen como mucho una llamada real por pregunta.

## 1. Sanity: imports (sin llamadas a red)

In [ ]:
import sys
from pathlib import Path

# Permite ejecutar el notebook desde la raiz del repo sin instalar el paquete.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "common").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Importar estos modulos NO carga el dataset ni crea clientes de red.
from common.responder import responder
from common.evaluar import evaluar

print("Imports OK: common.responder.responder, common.evaluar.evaluar")

## 2. Sanity: el dataset se resuelve correctamente (sin llamadas a red)

In [ ]:
from common.config import get_dataset_paths

paths = get_dataset_paths()
print("Dataset resuelto en:", paths.dataset_dir)
for name, path in paths.required_files().items():
    estado = "OK" if path.is_file() else "FALTA"
    print(f"  [{estado}] {name} -> {path}")

## 3. Sanity: contrato publico de las cuatro tools (sin llamadas a red)

In [ ]:
from common.tools import get_xbrl_fact, list_available, read_section, search_filings

for tool in (list_available, get_xbrl_fact, search_filings, read_section):
    print(f"{tool.name}{tuple(tool.args.keys())}")

## 4. Ejemplo: `responder(pregunta)`

Esta celda hace una llamada real a OpenRouter (requiere `OPENROUTER_API_KEY`).

In [ ]:
respuesta = responder(
    "¿Cuál fue el total de activos (Assets) de NVIDIA en el ejercicio fiscal 2024?"
)
respuesta

## 5. Ejemplo: `evaluar(ruta_jsonl)`

Sustituye `ruta_preguntas` por el archivo JSONL con tus preguntas (cada linea un JSON con, como minimo, `id`, `familia`, `pregunta` -mismo esquema que `common/golden_set/golden_set_grupo3.jsonl`-). Por defecto esta celda crea un archivo de una sola pregunta de ejemplo para no encarecer la ejecucion; `evaluar` acepta cualquier numero de preguntas nuevas sin tocar codigo.

In [ ]:
import json

# --- Reemplaza este bloque por tu archivo real de preguntas ---
ejemplo = {
    "id": "demo-001",
    "familia": "numerica",
    "pregunta": (
        "¿Cuál fue el total de activos (Assets) de NVIDIA en el "
        "ejercicio fiscal 2024?"
    ),
}
ruta_preguntas = Path("demo_preguntas.jsonl")
ruta_preguntas.write_text(
    json.dumps(ejemplo, ensure_ascii=False) + "\n", encoding="utf-8"
)
# --- fin del bloque de ejemplo ---

resultados = evaluar(str(ruta_preguntas), guardar_en="resultados_evaluacion.jsonl")
print(f"{len(resultados)} fila(s) evaluada(s). Guardado en resultados_evaluacion.jsonl")
resultados

## 6. Tests offline (opcional, desde terminal)

```powershell
python -m unittest discover -s common\tests -v
```

No hace llamadas a red y no depende de `OPENROUTER_API_KEY`.